# 🧠 Train Virgo-1.0-Angkor 12B — Google Vertex AI (Colab Enterprise)

The same Virgo 12B training as the Kaggle notebook, on a **Google Cloud L4 GPU** (24 GB, about 2–3× faster than Kaggle's T4, 32 GB RAM). A run takes about **1–2 hours** and costs roughly **$1–2** (on-demand L4 ≈ $0.85–1/hour; less with Spot). Prices change: check Google Cloud pricing.

Before the first run, open [google/gemma-3-12b-it](https://huggingface.co/google/gemma-3-12b-it) on Hugging Face and click **Agree and access**.

**First time only (Google Cloud)**
1. Create a project at [console.cloud.google.com](https://console.cloud.google.com) and turn on billing (new accounts get **$300 free credit**).
2. **IAM & Admin → Quotas**: request **1 × NVIDIA L4 GPU** (`NVIDIA_L4_GPUS`) in a region such as `us-central1`. New projects start with 0.
3. Turn on the **Vertex AI API** and the **Secret Manager API**.
4. **Security → Secret Manager → Create secret** for each secret below (name exactly as written, value = the token). Grant your Colab runtime's service account the **Secret Manager Secret Accessor** role. (Or skip this: the notebook asks for each one when it runs.)
5. **Vertex AI → Colab Enterprise → Runtime templates → New**: machine **g2-standard-8** (1 × L4, 32 GB RAM), disk 200 GB. Tick **Spot** to pay ~60–70% less (it can be stopped; just run again).

**Run**: Colab Enterprise → **Import notebook** from GitHub (`nangmrr752/virgo-models`, this file) → connect to your template's runtime → **Run all**.

**Secrets:** `HF_TOKEN` (Hugging Face token with **write** access).

**Run in the background:** Colab Enterprise → **Executions → Schedule / Run** this notebook on your template: it runs without the tab open. Use Secret Manager for `HF_TOKEN` (there's nobody to type it), and set `ASK_FOR_SECRETS = False` in Settings.

When it finishes it saves **Virgo-1.0-Angkor-12B** privately to your Hugging Face account, and the server notebooks use it automatically. Send the score table and sample answers to Claude. **Stop or delete the runtime afterwards** so it doesn't keep costing money.

In [ ]:
# ⚙️ Settings
BASE_MODEL = "google/gemma-3-12b-it"
MODEL_NAME = "Virgo-1.0-Angkor-12B"
EPOCHS = 3
OUT = "chat/out/virgo-1.0-angkor-12b"
ASK_FOR_SECRETS = True  # False for background Executions (use Secret Manager)

In [ ]:
# 1️⃣ Set up
import os, subprocess, sys, torch
assert torch.cuda.is_available(), "❌ No GPU: connect to a runtime template with an NVIDIA L4 (see the steps above)"
print("✅ GPUs:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
WORK = "/content" if os.path.isdir("/content") else os.path.expanduser("~")
os.chdir(WORK)
if not os.path.isdir("virgo-models/.git"):
    r = subprocess.run(["git", "clone", "-q", "https://github.com/nangmrr752/virgo-models"])
    assert r.returncode == 0, "❌ Can't download the Virgo code from GitHub"
os.chdir(f"{WORK}/virgo-models")
subprocess.run(["git", "pull", "-q"])
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-train.txt"], check=True)
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])  # conflicts with newer peft
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "google-cloud-secret-manager"])
def secret(name):
    """A secret: from the environment, else Google Secret Manager (this project), else asked once."""
    if os.environ.get(name):
        return os.environ[name]
    try:
        import google.auth
        from google.cloud import secretmanager
        _, project = google.auth.default()
        value = secretmanager.SecretManagerServiceClient().access_secret_version(
            name=f"projects/{project}/secrets/{name}/versions/latest").payload.data.decode().strip()
        os.environ[name] = value
        return value
    except Exception:
        pass
    if ASK_FOR_SECRETS:
        from getpass import getpass
        value = getpass(f"{name} (leave empty to skip): ").strip()
        if value:
            os.environ[name] = value
        return value or None
    return None

from huggingface_hub import login, HfApi
token = secret("HF_TOKEN")
assert token, "❌ No HF_TOKEN: add it in Secret Manager (see the steps above) or type it when asked"
login(token=token)
REPO = f"{HfApi().whoami()['name']}/{MODEL_NAME}"
print("✅ Ready. Virgo 12B will be saved to", REPO)

def run(cmd):
    """Runs a step and shows its output here; on failure, shows the last lines (send them to Claude)."""
    import subprocess, collections
    tail = collections.deque(maxlen=60)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
        tail.append(line)
    if proc.wait() != 0:
        print("\n" + "=" * 60 + "\n❌ It stopped with an error. Send these last lines to Claude:\n" + "=" * 60)
        print("".join(tail))
        raise SystemExit(f"❌ {next((a for a in cmd if a.endswith('.py')), cmd[0])} failed (exit code {proc.returncode})")

In [ ]:
# 2️⃣ Train Virgo on Gemma 3 12B
import subprocess, sys
run([sys.executable, "scripts/check_data.py"])
run([sys.executable, "-u", "chat/train.py", "--base", BASE_MODEL, "--epochs", str(EPOCHS), "--out", OUT])

In [ ]:
# 3️⃣ Save Virgo-1.0-Angkor-12B to your Hugging Face account (private)
from huggingface_hub import HfApi
card = f"""---
base_model: {BASE_MODEL}
library_name: peft
license: gemma
language: [en, km]
tags: [virgo, ksn, khmer, lora]
---
# {MODEL_NAME}

**{MODEL_NAME}** is KSN's own assistant model for [Virgo AI](https://virgoai.camksn.com): friendly, clear and honest, in English and Khmer.

It is a LoRA fine-tune of Google's [{BASE_MODEL}](https://huggingface.co/{BASE_MODEL}), trained on Virgo's own conversations.
Use is subject to the [Gemma Terms of Use](https://ai.google.dev/gemma/terms).

Made by KSN · https://soknang.camksn.com/
"""
open(f"{OUT}/README.md", "w", encoding="utf-8").write(card)
api = HfApi()
api.create_repo(REPO, private=True, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=REPO, commit_message=MODEL_NAME, ignore_patterns=["checkpoint-*"])
print(f"✅ Saved {MODEL_NAME}: https://huggingface.co/{REPO}")

In [ ]:
# 4️⃣ Score Virgo 12B (send this table to Claude)
import subprocess, sys
run([sys.executable, "chat/evaluate.py", "--adapter", OUT])

In [ ]:
# 5️⃣ Try Virgo-1.0-Angkor 12B
import sys; sys.path.append("chat")
from virgo_chat import VirgoChat
bot = VirgoChat(adapter=OUT)
for q in ["Who are you?", "អ្នកជានរណា?", "What is 15% of 200?", "ពន្យល់ AI ឱ្យងាយយល់", "Translate to Khmer: Thank you for your help."]:
    print("🧑", q)
    print("🤖", bot.reply([{"role": "user", "content": q}], temperature=0), "\n")
print("🎉 Done! Send the score table and these answers to Claude.")